# Capstone Part 5: Serving

In this notebook we build a production-style serving layer for our model:

1. **FastAPI wrapper** with request/response API
2. **Streaming endpoint** using Server-Sent Events (SSE)
3. **Load testing** with concurrent requests
4. **Throughput and latency measurement**
5. **vLLM deployment configuration**

We run the server in a background thread within the notebook, then test it
with HTTP requests.

## 1. Environment Setup

In [ ]:
!pip install -q torch tiktoken fastapi uvicorn httpx matplotlib tqdm

In [ ]:
import math
import time
import json
import threading
import asyncio
from pathlib import Path
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed

import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load Model

In [ ]:
# ---- Model definition (same as previous notebooks) ----

@dataclass
class ModelConfig:
    vocab_size: int = 50257
    max_seq_len: int = 512
    d_model: int = 256
    n_layers: int = 8
    n_heads: int = 8
    n_kv_heads: int = 2
    d_ff: int = 688
    dropout: float = 0.1
    rope_theta: float = 10000.0


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return (x * (x.float().pow(2).mean(-1, keepdim=True) + self.eps).rsqrt()).type_as(x) * self.weight


def precompute_rope_frequencies(dim, max_seq_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    angles = torch.outer(torch.arange(max_seq_len).float(), freqs)
    return torch.polar(torch.ones_like(angles), angles)


def apply_rope(x, freqs_cis):
    b, s, n, d = x.shape
    x_c = torch.view_as_complex(x.float().reshape(b, s, n, -1, 2))
    fc = freqs_cis[:s].unsqueeze(0).unsqueeze(2)
    return torch.view_as_real(x_c * fc).reshape(b, s, n, d).type_as(x)


class GroupedQueryAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads, self.n_kv_heads = config.n_heads, config.n_kv_heads
        self.head_dim = config.d_model // config.n_heads
        self.n_rep = self.n_heads // self.n_kv_heads
        self.wq = nn.Linear(config.d_model, config.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(config.d_model, config.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(config.n_heads * self.head_dim, config.d_model, bias=False)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
    def _repeat_kv(self, x):
        if self.n_rep == 1: return x
        b, s, n, d = x.shape
        return x[:,:,:,None,:].expand(b,s,n,self.n_rep,d).reshape(b,s,self.n_heads,d)
    def forward(self, x, freqs_cis, mask=None):
        b, s, _ = x.shape
        q = self.wq(x).reshape(b,s,self.n_heads,self.head_dim)
        k = self.wk(x).reshape(b,s,self.n_kv_heads,self.head_dim)
        v = self.wv(x).reshape(b,s,self.n_kv_heads,self.head_dim)
        q, k = apply_rope(q, freqs_cis), apply_rope(k, freqs_cis)
        k, v = self._repeat_kv(k), self._repeat_kv(v)
        q, k, v = q.transpose(1,2), k.transpose(1,2), v.transpose(1,2)
        attn = torch.matmul(q, k.transpose(-2,-1)) / math.sqrt(self.head_dim)
        if mask is not None: attn = attn.masked_fill(mask == 0, float("-inf"))
        out = torch.matmul(self.attn_dropout(F.softmax(attn, dim=-1)), v)
        return self.resid_dropout(self.wo(out.transpose(1,2).reshape(b,s,-1)))


class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.w1 = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.w2 = nn.Linear(config.d_ff, config.d_model, bias=False)
        self.w3 = nn.Linear(config.d_model, config.d_ff, bias=False)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention_norm = RMSNorm(config.d_model)
        self.attention = GroupedQueryAttention(config)
        self.ffn_norm = RMSNorm(config.d_model)
        self.ffn = SwiGLU(config)
    def forward(self, x, freqs_cis, mask=None):
        x = x + self.attention(self.attention_norm(x), freqs_cis, mask)
        return x + self.ffn(self.ffn_norm(x))


class MiniLLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        self.layers = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.norm = RMSNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        head_dim = config.d_model // config.n_heads
        self.register_buffer("freqs_cis", precompute_rope_frequencies(head_dim, config.max_seq_len, config.rope_theta), persistent=False)
        self.apply(self._init_weights)
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    def forward(self, input_ids, targets=None):
        b, s = input_ids.shape
        mask = torch.tril(torch.ones(s, s, device=input_ids.device)).unsqueeze(0).unsqueeze(0)
        x = self.dropout(self.tok_emb(input_ids))
        for layer in self.layers:
            x = layer(x, self.freqs_cis, mask)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=100, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx = input_ids[:, -self.config.max_seq_len:]
            logits, _ = self(idx)
            logits = logits[:, -1, :] / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            input_ids = torch.cat([input_ids, torch.multinomial(F.softmax(logits, dim=-1), 1)], dim=1)
        return input_ids

    @torch.no_grad()
    def generate_stream(self, input_ids, max_new_tokens=100, temperature=0.8, top_k=50):
        """Generator that yields one token at a time for streaming."""
        for _ in range(max_new_tokens):
            idx = input_ids[:, -self.config.max_seq_len:]
            logits, _ = self(idx)
            logits = logits[:, -1, :] / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            next_token = torch.multinomial(F.softmax(logits, dim=-1), 1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            yield next_token.item()


print("Model architecture with streaming support defined.")

In [ ]:
save_dir = Path("checkpoints")
checkpoint = torch.load(save_dir / "model_for_serving.pt", map_location=device, weights_only=False)

config = checkpoint["config"]
tokenizer = tiktoken.get_encoding("gpt2")

model = MiniLLM(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Model loaded on {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. FastAPI Application

We define a FastAPI app with two endpoints:
- `/v1/completions` -- Standard request/response
- `/v1/completions/stream` -- Server-Sent Events (SSE) streaming

The API follows OpenAI's format for compatibility.

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse, JSONResponse
from pydantic import BaseModel, Field
import uvicorn

INSTRUCTION_PREFIX = "<|instruction|> "
RESPONSE_PREFIX = " <|response|> "
END_TOKEN = "<|end|>"


class CompletionRequest(BaseModel):
    """Request schema matching OpenAI's API format."""
    prompt: str
    max_tokens: int = Field(default=100, ge=1, le=512)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    top_k: int = Field(default=50, ge=0, le=100)
    stream: bool = False


class CompletionResponse(BaseModel):
    """Response schema."""
    id: str
    object: str = "text_completion"
    created: int
    model: str = "mini-llm-10m"
    choices: list
    usage: dict


# Create the FastAPI app
app = FastAPI(
    title="MiniLLM Serving API",
    description="Serve our pretrained + SFT + DPO model via HTTP",
    version="1.0.0",
)

# Request counter for unique IDs
request_counter = 0


@app.get("/health")
def health_check():
    return {"status": "healthy", "model": "mini-llm-10m", "device": str(device)}


@app.post("/v1/completions")
def create_completion(request: CompletionRequest):
    global request_counter
    request_counter += 1

    start_time = time.time()

    # Format prompt
    formatted = INSTRUCTION_PREFIX + request.prompt + RESPONSE_PREFIX
    input_ids = torch.tensor([tokenizer.encode(formatted)], device=device)
    prompt_tokens = input_ids.shape[1]

    if request.stream:
        # Return streaming response
        def generate_sse():
            for token_id in model.generate_stream(
                input_ids.clone(),
                max_new_tokens=request.max_tokens,
                temperature=max(request.temperature, 0.01),
                top_k=request.top_k,
            ):
                token_str = tokenizer.decode([token_id])
                if END_TOKEN in token_str:
                    break
                chunk = {
                    "id": f"cmpl-{request_counter}",
                    "object": "text_completion.chunk",
                    "choices": [{"text": token_str, "index": 0}],
                }
                yield f"data: {json.dumps(chunk)}\n\n"
            yield "data: [DONE]\n\n"

        return StreamingResponse(generate_sse(), media_type="text/event-stream")

    # Non-streaming: generate all at once
    output_ids = model.generate(
        input_ids,
        max_new_tokens=request.max_tokens,
        temperature=max(request.temperature, 0.01),
        top_k=request.top_k,
    )

    # Decode and clean response
    full_text = tokenizer.decode(output_ids[0].tolist())
    if "<|response|>" in full_text:
        response_text = full_text.split("<|response|>")[-1]
        if END_TOKEN in response_text:
            response_text = response_text.split(END_TOKEN)[0]
        response_text = response_text.strip()
    else:
        response_text = full_text[len(formatted):].strip()

    completion_tokens = output_ids.shape[1] - prompt_tokens
    elapsed = time.time() - start_time

    return CompletionResponse(
        id=f"cmpl-{request_counter}",
        created=int(time.time()),
        choices=[{
            "text": response_text,
            "index": 0,
            "finish_reason": "stop" if END_TOKEN in full_text else "length",
        }],
        usage={
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": prompt_tokens + completion_tokens,
            "latency_ms": round(elapsed * 1000, 2),
        },
    )


print("FastAPI application defined with /health and /v1/completions endpoints.")

## 4. Launch Server in Background Thread

We run the uvicorn server in a background thread so we can interact with it
from subsequent cells.

In [ ]:
PORT = 8234

server_config = uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning")
server = uvicorn.Server(server_config)

# Run in background thread
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(2)
print(f"Server running at http://127.0.0.1:{PORT}")
print(f"Endpoints:")
print(f"  GET  /health")
print(f"  POST /v1/completions")

## 5. Test Health Endpoint

In [ ]:
import httpx

BASE_URL = f"http://127.0.0.1:{PORT}"

# Health check
resp = httpx.get(f"{BASE_URL}/health")
print(f"Health check: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

## 6. Test Completion Endpoint

In [ ]:
# Single completion request
resp = httpx.post(
    f"{BASE_URL}/v1/completions",
    json={
        "prompt": "Write a short story about a cat.",
        "max_tokens": 80,
        "temperature": 0.7,
    },
    timeout=30.0,
)

print(f"Status: {resp.status_code}")
result = resp.json()
print(f"\nResponse:")
print(json.dumps(result, indent=2))

## 7. Test Streaming Endpoint

Streaming delivers tokens one at a time via Server-Sent Events (SSE).
This provides a much better user experience for long generations.

In [ ]:
# Streaming request
print("Streaming response:")
print("-" * 50)

with httpx.stream(
    "POST",
    f"{BASE_URL}/v1/completions",
    json={
        "prompt": "Tell a story about a brave little mouse.",
        "max_tokens": 60,
        "temperature": 0.7,
        "stream": True,
    },
    timeout=30.0,
) as response:
    full_text = ""
    for line in response.iter_lines():
        if line.startswith("data: "):
            data = line[6:]
            if data == "[DONE]":
                break
            chunk = json.loads(data)
            token = chunk["choices"][0]["text"]
            full_text += token
            print(token, end="", flush=True)

print("\n" + "-" * 50)
print(f"\nTotal streamed text length: {len(full_text)} characters")

## 8. Latency Benchmarking

Measure end-to-end latency for different request sizes.

In [ ]:
def measure_latency(prompt, max_tokens, n_runs=5):
    """Measure average latency for a completion request."""
    latencies = []
    for _ in range(n_runs):
        start = time.time()
        resp = httpx.post(
            f"{BASE_URL}/v1/completions",
            json={"prompt": prompt, "max_tokens": max_tokens, "temperature": 0.7},
            timeout=60.0,
        )
        elapsed = time.time() - start
        latencies.append(elapsed)
    return {
        "avg_ms": sum(latencies) / len(latencies) * 1000,
        "min_ms": min(latencies) * 1000,
        "max_ms": max(latencies) * 1000,
        "p50_ms": sorted(latencies)[len(latencies)//2] * 1000,
    }


token_counts = [10, 25, 50, 100]
latency_results = []

print("Measuring latency for different output lengths...")
for max_tok in tqdm(token_counts):
    result = measure_latency("Write a story.", max_tok, n_runs=3)
    result["max_tokens"] = max_tok
    latency_results.append(result)
    print(f"  {max_tok} tokens: avg={result['avg_ms']:.0f}ms, p50={result['p50_ms']:.0f}ms")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
toks = [r["max_tokens"] for r in latency_results]
avgs = [r["avg_ms"] for r in latency_results]
mins = [r["min_ms"] for r in latency_results]
maxs = [r["max_ms"] for r in latency_results]

ax.plot(toks, avgs, 'bo-', linewidth=2, markersize=8, label='Average')
ax.fill_between(toks, mins, maxs, alpha=0.2, color='blue', label='Min-Max range')
ax.set_xlabel('Max Output Tokens')
ax.set_ylabel('Latency (ms)')
ax.set_title('End-to-End Latency vs Output Length')
ax.legend()
ax.grid(True, alpha=0.3)

# Add tokens/sec annotation
for tok, avg in zip(toks, avgs):
    tps = tok / (avg / 1000)
    ax.annotate(f'{tps:.0f} tok/s', (tok, avg), textcoords="offset points",
                xytext=(10, 10), fontsize=9)

plt.tight_layout()
plt.show()

## 9. Time to First Token (TTFT)

TTFT measures how quickly the first token arrives when streaming.
This is the key UX metric for interactive applications.

In [ ]:
def measure_ttft(prompt, n_runs=5):
    """Measure Time To First Token using streaming."""
    ttfts = []
    for _ in range(n_runs):
        start = time.time()
        with httpx.stream(
            "POST",
            f"{BASE_URL}/v1/completions",
            json={"prompt": prompt, "max_tokens": 20, "temperature": 0.7, "stream": True},
            timeout=30.0,
        ) as response:
            for line in response.iter_lines():
                if line.startswith("data: ") and line[6:] != "[DONE]":
                    ttft = time.time() - start
                    ttfts.append(ttft)
                    break
    return sum(ttfts) / len(ttfts) * 1000 if ttfts else 0


prompts_by_length = [
    ("Short", "Hi"),
    ("Medium", "Write a short story about a cat."),
    ("Long", "Write a detailed and engaging story about a brave little dog who goes on an adventure to find a lost treasure hidden in a magical forest."),
]

print("Measuring Time To First Token (TTFT):")
ttft_results = []
for label, prompt in prompts_by_length:
    ttft = measure_ttft(prompt, n_runs=3)
    ttft_results.append((label, ttft))
    print(f"  {label} prompt ({len(tokenizer.encode(prompt))} tokens): TTFT = {ttft:.0f}ms")

## 10. Load Testing with Concurrent Requests

Production systems must handle multiple simultaneous requests. Let's measure
how throughput and latency change under load.

In [ ]:
def send_request(prompt, max_tokens=30):
    """Send a single completion request and return timing info."""
    start = time.time()
    try:
        resp = httpx.post(
            f"{BASE_URL}/v1/completions",
            json={"prompt": prompt, "max_tokens": max_tokens, "temperature": 0.7},
            timeout=60.0,
        )
        elapsed = time.time() - start
        return {
            "status": resp.status_code,
            "latency_ms": elapsed * 1000,
            "tokens": resp.json().get("usage", {}).get("completion_tokens", 0),
            "success": resp.status_code == 200,
        }
    except Exception as e:
        return {
            "status": 0,
            "latency_ms": (time.time() - start) * 1000,
            "tokens": 0,
            "success": False,
            "error": str(e),
        }


test_prompts = [
    "Write about a sunny day.",
    "Tell a story about a bird.",
    "Write about friendship.",
    "Tell a bedtime story.",
    "Write about the ocean.",
    "Tell a story about cooking.",
    "Write about a garden.",
    "Tell a story about rain.",
]

concurrency_levels = [1, 2, 4]
load_test_results = []

print("Running load test...")
for n_concurrent in concurrency_levels:
    results = []
    total_start = time.time()

    with ThreadPoolExecutor(max_workers=n_concurrent) as executor:
        futures = [
            executor.submit(send_request, test_prompts[i % len(test_prompts)])
            for i in range(8)
        ]
        for future in as_completed(futures):
            results.append(future.result())

    total_time = time.time() - total_start
    successful = [r for r in results if r["success"]]
    avg_latency = sum(r["latency_ms"] for r in successful) / len(successful) if successful else 0
    total_tokens = sum(r["tokens"] for r in successful)
    throughput = total_tokens / total_time if total_time > 0 else 0

    summary = {
        "concurrency": n_concurrent,
        "requests": len(results),
        "successful": len(successful),
        "avg_latency_ms": avg_latency,
        "throughput_tok_s": throughput,
        "total_time_s": total_time,
    }
    load_test_results.append(summary)

    print(
        f"  Concurrency {n_concurrent}: "
        f"{len(successful)}/{len(results)} OK | "
        f"Avg latency: {avg_latency:.0f}ms | "
        f"Throughput: {throughput:.1f} tok/s | "
        f"Total: {total_time:.1f}s"
    )

In [ ]:
# Visualize load test results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

concurrencies = [r["concurrency"] for r in load_test_results]

# Latency
ax = axes[0]
latencies = [r["avg_latency_ms"] for r in load_test_results]
ax.bar(concurrencies, latencies, color='coral', width=0.5)
ax.set_xlabel('Concurrent Requests')
ax.set_ylabel('Avg Latency (ms)')
ax.set_title('Latency vs Concurrency')
ax.set_xticks(concurrencies)
ax.grid(True, alpha=0.3, axis='y')

# Throughput
ax = axes[1]
throughputs = [r["throughput_tok_s"] for r in load_test_results]
ax.bar(concurrencies, throughputs, color='steelblue', width=0.5)
ax.set_xlabel('Concurrent Requests')
ax.set_ylabel('Throughput (tok/s)')
ax.set_title('Throughput vs Concurrency')
ax.set_xticks(concurrencies)
ax.grid(True, alpha=0.3, axis='y')

# Total wall time
ax = axes[2]
times = [r["total_time_s"] for r in load_test_results]
ax.bar(concurrencies, times, color='mediumpurple', width=0.5)
ax.set_xlabel('Concurrent Requests')
ax.set_ylabel('Total Time (s)')
ax.set_title('Wall Time for 8 Requests')
ax.set_xticks(concurrencies)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Load Test Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(save_dir / 'serving_load_test.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Request/Response Logging and Monitoring

In production, you need observability. Here we demonstrate how to add
request logging and basic metrics collection.

In [ ]:
# Simulate a monitoring dashboard with request history
# (In production, use Prometheus + Grafana or similar)

# Send a batch of diverse requests and collect metrics
monitoring_data = []

diverse_prompts = [
    ("Write a story about a cat.", 50),
    ("Tell me about the sun.", 30),
    ("Write a long story about a magical forest with many animals.", 100),
    ("Hi", 20),
    ("Write about winter.", 40),
    ("Tell a story.", 60),
]

print("Collecting monitoring data...")
for prompt, max_tok in diverse_prompts:
    start = time.time()
    resp = httpx.post(
        f"{BASE_URL}/v1/completions",
        json={"prompt": prompt, "max_tokens": max_tok, "temperature": 0.7},
        timeout=60.0,
    )
    elapsed = time.time() - start
    data = resp.json()
    monitoring_data.append({
        "prompt_tokens": data["usage"]["prompt_tokens"],
        "completion_tokens": data["usage"]["completion_tokens"],
        "latency_ms": elapsed * 1000,
        "tokens_per_sec": data["usage"]["completion_tokens"] / elapsed,
    })

# Monitoring dashboard
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

req_ids = range(len(monitoring_data))

# Latency per request
ax = axes[0, 0]
latencies = [d["latency_ms"] for d in monitoring_data]
ax.bar(req_ids, latencies, color='coral')
ax.set_xlabel('Request #')
ax.set_ylabel('Latency (ms)')
ax.set_title('Per-Request Latency')
ax.grid(True, alpha=0.3, axis='y')

# Tokens per second
ax = axes[0, 1]
tps = [d["tokens_per_sec"] for d in monitoring_data]
ax.bar(req_ids, tps, color='steelblue')
ax.set_xlabel('Request #')
ax.set_ylabel('Tokens/Second')
ax.set_title('Per-Request Throughput')
ax.grid(True, alpha=0.3, axis='y')

# Token counts
ax = axes[1, 0]
prompt_toks = [d["prompt_tokens"] for d in monitoring_data]
comp_toks = [d["completion_tokens"] for d in monitoring_data]
ax.bar(req_ids, prompt_toks, label='Prompt', color='lightblue')
ax.bar(req_ids, comp_toks, bottom=prompt_toks, label='Completion', color='navy')
ax.set_xlabel('Request #')
ax.set_ylabel('Tokens')
ax.set_title('Token Distribution')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Latency vs completion tokens (should be linear)
ax = axes[1, 1]
ax.scatter(comp_toks, latencies, s=80, color='purple', alpha=0.7)
ax.set_xlabel('Completion Tokens')
ax.set_ylabel('Latency (ms)')
ax.set_title('Latency vs Output Length')
ax.grid(True, alpha=0.3)

plt.suptitle('Serving Monitoring Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. vLLM Deployment Configuration

For production serving at scale, **vLLM** is the state-of-the-art serving engine.
It provides:
- **PagedAttention**: Efficient KV-cache management
- **Continuous batching**: Dynamic batching of incoming requests
- **Tensor parallelism**: Multi-GPU serving

Below is a reference configuration for deploying our model with vLLM.
Note: Our custom model would need a vLLM-compatible wrapper in production.

In [ ]:
# vLLM deployment configuration (reference)
vllm_config = {
    "model": "./checkpoints/model_for_serving.pt",
    "tokenizer": "gpt2",
    "tensor_parallel_size": 1,
    "gpu_memory_utilization": 0.85,
    "max_model_len": config.max_seq_len,
    "max_num_seqs": 32,
    "max_num_batched_tokens": 4096,
    "dtype": "auto",
    "quantization": None,  # Could use "awq" or "gptq" for quantized models
    "enforce_eager": False,  # Use CUDA graphs when possible
    "swap_space": 4,  # GB of CPU memory for KV-cache swapping
}

print("vLLM Deployment Configuration:")
print(json.dumps(vllm_config, indent=2))

print("\n--- Example vLLM launch command ---")
print("""
# For a HuggingFace-compatible model:
python -m vllm.entrypoints.openai.api_server \\
    --model ./model_directory \\
    --tokenizer gpt2 \\
    --host 0.0.0.0 \\
    --port 8000 \\
    --max-model-len 512 \\
    --gpu-memory-utilization 0.85 \\
    --dtype auto
""")

print("--- Docker deployment ---")
print("""
FROM vllm/vllm-openai:latest
COPY ./checkpoints /model
CMD ["--model", "/model", "--port", "8000"]
""")

## 13. Shutdown Server

In [ ]:
# Graceful shutdown
server.should_exit = True
time.sleep(1)
print("Server shut down.")

## 14. Serving Metrics Summary

In [ ]:
print("=" * 60)
print("SERVING METRICS SUMMARY")
print("=" * 60)
print(f"\nModel: MiniLLM ~10M params")
print(f"Device: {device}")
print(f"\nLatency Benchmarks:")
for r in latency_results:
    print(f"  {r['max_tokens']} tokens: avg={r['avg_ms']:.0f}ms")

print(f"\nLoad Test Results:")
for r in load_test_results:
    print(
        f"  Concurrency {r['concurrency']}: "
        f"latency={r['avg_latency_ms']:.0f}ms, "
        f"throughput={r['throughput_tok_s']:.1f} tok/s"
    )

print(f"\nTTFT:")
for label, ttft in ttft_results:
    print(f"  {label} prompt: {ttft:.0f}ms")

## Summary

In this notebook we:

1. **Built** a FastAPI serving wrapper with OpenAI-compatible API
2. **Implemented** streaming via Server-Sent Events (SSE)
3. **Benchmarked** latency across different output lengths
4. **Measured** Time To First Token (TTFT) for streaming UX
5. **Load tested** with concurrent requests to measure throughput
6. **Provided** vLLM deployment configuration for production scale

Key takeaways:
- Latency scales linearly with output tokens (autoregressive generation)
- TTFT is dominated by prompt encoding time
- Without batching optimization (like vLLM), concurrent requests serialize
- Production serving needs KV-cache management and continuous batching

In the final notebook, we will evaluate our model at each stage of the pipeline.